# Comprendre les Buffers(tampons) PyTorch

En substance, les buffers PyTorch sont des attributs tensoriels associés à un module ou à un modèle PyTorch, de la même manière que les paramètres. Toutefois, contrairement aux paramètres, les buffers ne sont pas mis à jour pendant l'entraînement.

Les buffers PyTorch sont particulièrement utiles lorsqu'on travaille avec des calculs sur GPU, car ils doivent être transférés d'un appareil à un autre, comme du CPU vers le GPU, en même temps que les paramètres du modèle. Contrairement aux paramètres, les buffers ne nécessitent pas le calcul de gradients, mais ils doivent tout de même se trouver sur le bon appareil pour que tous les calculs soient effectués correctement.

Dans le chapitre 3, nous utilisons les buffers PyTorch via `self.register_buffer`, une fonction seulement expliquée brièvement dans le livre. Comme ce concept et son utilité ne sont pas immédiatement évidents, ce notebook propose une explication plus détaillée accompagnée d'un exemple pratique.

## Un exemple sans buffers

Supposons que nous ayons le code suivant, inspiré du chapitre 3. Cette version a été modifiée pour exclure les buffers. Elle implémente le mécanisme d'auto-attention causal utilisé dans les LLM :

In [1]:
import torch
import torch.nn as nn

class CausalAttentionWithoutBuffers(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights  = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values

        return context_vec

Nous pouvons initialiser et exécuter le module comme suit sur des données d'exemple :

In [2]:
torch.manual_seed(123)

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

batch = torch.stack([inputs, inputs], dim=0)  # (batch, num_tokens, d_in)
context_length = batch.shape[1]
d_in = batch.shape[2]
d_out = 2

ca_without_buffer = CausalAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)

with torch.no_grad():
    context_vecs = ca_without_buffer(batch)

print(context_vecs)


tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]])


Jusqu'ici, tout a bien fonctionné.

Cependant, lors de l'entraînement des LLM, nous utilisons généralement des GPU pour accélérer le processus. Par conséquent, transférons le module CausalAttentionWithoutBuffers vers un périphérique GPU.

Veuillez noter que cette opération nécessite d'exécuter le code dans un environnement équipé de GPU.

In [3]:
has_cuda = torch.cuda.is_available()
has_mps = torch.backends.mps.is_available()

print("Machine has GPU:", has_cuda or has_mps)

if has_mps:
    device = torch.device("mps")  # Apple Sillicon GPU (Metal)
elif has_cuda:
    device = torch.device("cuda") # NVIDIA GPU (CUDA)
else:
    device = torch.device("cpu")  # CPU fallback

print(f"Using device: {device}")
batch = batch.to(device)
ca_without_buffer = ca_without_buffer.to(device)

Machine has GPU: True
Using device: cuda


Maintenant, exécutons à nouveau le code :

In [4]:
with torch.no_grad():
  context_vecs = ca_without_buffer(batch)
print(context_vecs)

RuntimeError: expected self and mask to be on the same device, but got mask on cpu and self on cuda:0

L'exécution du code a entraîné une erreur. Que s'est-il passé ? Il semble que nous ayons tenté d'effectuer une multiplication matricielle entre un tenseur situé sur le **GPU** et un autre situé sur le **CPU**. Pourtant, nous avons bien déplacé le module vers le GPU !

Vérifions à nouveau sur quels dispositifs (*devices*) se trouvent certains tenseurs :


In [5]:
print("W_query.device:", ca_without_buffer.W_query.weight.device)
print("mask.device:", ca_without_buffer.mask.device)

W_query.device: cuda:0
mask.device: cpu


In [6]:
type(ca_without_buffer.mask)

torch.Tensor

Comme nous pouvons le voir, le tenseur `mask` n'a pas été déplacé vers le **GPU**. Cela s'explique par le fait qu'il ne s'agit pas d'un paramètre PyTorch comme les poids (par exemple, `W_query.weight`).

Cela signifie que nous devons le déplacer manuellement vers le GPU à l'aide de `.to("cuda")` :


In [7]:
ca_without_buffer.mask = ca_without_buffer.mask.to(device)
print("mas.device:", ca_without_buffer.mask.device)

mas.device: cuda:0


Essayons à nouveau notre code :


In [8]:
with torch.no_grad():
  context_vecs = ca_without_buffer(batch)

print(context_vecs)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], device='cuda:0')


Cette fois, cela a fonctionné !

Cependant, devoir penser à déplacer manuellement chaque tenseur vers le GPU peut être fastidieux. Comme nous le verrons dans la section suivante, il est plus pratique d'utiliser `register_buffer` pour enregistrer le tenseur `mask` comme un **buffer**.

## Un exemple avec les buffers

Modifions maintenant la classe d'attention causale afin d'enregistrer le `mask` causal comme un buffer :

In [9]:
import torch
import torch.nn as nn

class CausalAttentionWithBuffer(nn.Module):
  def __init__(self, d_in, d_out, context_length,
               dropout, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    self.dropout = nn.Dropout(dropout)

    # Old:
    # self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    # New:
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vecs = attn_weights @ values

    return context_vecs

Désormais, de manière pratique, si nous déplaçons le module vers le GPU, le masque (`mask`) sera lui aussi automatiquement déplacé sur le GPU :


In [10]:
ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
ca_with_buffer.to(device)

print("W_query.device:", ca_with_buffer.W_query.weight.device)
print("mask.device:", ca_with_buffer.mask.device)

W_query.device: cuda:0
mask.device: cuda:0


In [11]:
with torch.no_grad():
  context_vecs = ca_with_buffer(batch)

print(context_vecs)

tensor([[[0.4772, 0.1063],
         [0.5891, 0.3257],
         [0.6202, 0.3860],
         [0.5478, 0.3589],
         [0.5321, 0.3428],
         [0.5077, 0.3493]],

        [[0.4772, 0.1063],
         [0.5891, 0.3257],
         [0.6202, 0.3860],
         [0.5478, 0.3589],
         [0.5321, 0.3428],
         [0.5077, 0.3493]]], device='cuda:0')


Comme nous pouvons le voir ci-dessus, enregistrer un tenseur comme un **buffer** peut nous simplifier la vie : nous n’avons plus besoin de penser à déplacer manuellement les tenseurs vers un périphérique cible comme un GPU.


## Buffers et `state_dict`

* Un autre avantage des buffers PyTorch, par rapport aux tenseurs classiques, est qu’ils sont inclus dans le `state_dict` d’un modèle.
* Par exemple, considérons le `state_dict` de l’objet d’attention causale sans buffers.


In [12]:
ca_without_buffer.state_dict()

OrderedDict([('W_query.weight',
              tensor([[-0.2354,  0.0191, -0.2867],
                      [ 0.2177, -0.4919,  0.4232]], device='cuda:0')),
             ('W_key.weight',
              tensor([[-0.4196, -0.4590, -0.3648],
                      [ 0.2615, -0.2133,  0.2161]], device='cuda:0')),
             ('W_value.weight',
              tensor([[-0.4900, -0.3503, -0.2120],
                      [-0.1135, -0.4404,  0.3780]], device='cuda:0'))])

* Le masque n’est pas inclus dans le `state_dict` ci-dessus.
* En revanche, le masque *est* bien inclus dans le `state_dict` ci-dessous, grâce à son enregistrement en tant que buffer.


In [13]:
ca_with_buffer.state_dict()

OrderedDict([('mask',
              tensor([[0., 1., 1., 1., 1., 1.],
                      [0., 0., 1., 1., 1., 1.],
                      [0., 0., 0., 1., 1., 1.],
                      [0., 0., 0., 0., 1., 1.],
                      [0., 0., 0., 0., 0., 1.],
                      [0., 0., 0., 0., 0., 0.]], device='cuda:0')),
             ('W_query.weight',
              tensor([[-0.1362,  0.1853,  0.4083],
                      [ 0.1076,  0.1579,  0.5573]], device='cuda:0')),
             ('W_key.weight',
              tensor([[-0.2604,  0.1829, -0.2569],
                      [ 0.4126,  0.4611, -0.5323]], device='cuda:0')),
             ('W_value.weight',
              tensor([[ 0.4929,  0.2757,  0.2516],
                      [ 0.2377,  0.4800, -0.0762]], device='cuda:0'))])

* Un `state_dict` est utile pour sauvegarder et recharger des modèles PyTorch entraînés, par exemple.
* Dans ce cas précis, sauvegarder et charger le `mask` n’est peut-être pas très utile, car il reste inchangé pendant l’entraînement ; cependant, à des fins de démonstration, supposons qu’il ait été modifié de sorte que tous les `1` aient été remplacés par des `2` :


In [14]:
ca_with_buffer.mask[ca_with_buffer.mask == 1.] = 2.
ca_with_buffer.mask

tensor([[0., 2., 2., 2., 2., 2.],
        [0., 0., 2., 2., 2., 2.],
        [0., 0., 0., 2., 2., 2.],
        [0., 0., 0., 0., 2., 2.],
        [0., 0., 0., 0., 0., 2.],
        [0., 0., 0., 0., 0., 0.]], device='cuda:0')

* Ensuite, si nous sauvegardons puis rechargeons le modèle, nous pouvons constater que le masque est restauré avec sa valeur modifiée.


In [15]:
torch.save(ca_with_buffer.state_dict(), "model.pth")

new_ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
new_ca_with_buffer.load_state_dict(torch.load("model.pth"))

new_ca_with_buffer.mask

tensor([[0., 2., 2., 2., 2., 2.],
        [0., 0., 2., 2., 2., 2.],
        [0., 0., 0., 2., 2., 2.],
        [0., 0., 0., 0., 2., 2.],
        [0., 0., 0., 0., 0., 2.],
        [0., 0., 0., 0., 0., 0.]])

* Cela n’est pas vrai si nous n’utilisons pas les buffers :


In [16]:
ca_without_buffer.mask[ca_without_buffer.mask == 1.] = 2.

torch.save(ca_without_buffer.state_dict(), "model.pth")

new_ca_without_buffer = CausalAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)
new_ca_without_buffer.load_state_dict(torch.load("model.pth"))

new_ca_without_buffer.mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])